# What Is Machine Learning?

## What you'll learn
- What machine learning is, and how it differs from writing rules by hand
- Supervised learning (regression and classification) and unsupervised learning
- Features, targets, and why you split data into training and test sets
- The scikit-learn pattern every model follows: create, `fit`, `predict`
- Why every model must beat a simple baseline

## Rules vs learning

In ordinary code you write the rules: "if days to pay is over 45, flag the account".
In **machine learning** you give the computer examples (past accounts and whether they
paid late) and it **learns** the rules from the data.

| Type | You have | The model learns to | Example in this stage |
|------|----------|---------------------|------------------------|
| Regression | Examples with a number to predict | Predict a number | Next year's spend |
| Classification | Examples with a label to predict | Predict a category | Will this account pay late? |
| Clustering | Examples with no label | Group similar rows | Customer segments |

Regression and classification are **supervised** (the answers are in the data).
Clustering is **unsupervised** (there is no answer column).

## The tools

- **scikit-learn** (`sklearn`): the standard library for classic machine learning. Lessons 1 to 7.
- **PyTorch** and **TensorFlow (Keras)**: libraries for neural networks. Lessons 8 and 9.

In [ ]:
import pandas as pd

df = pd.read_csv("data/customer_accounts.csv")

# Quick tidy-up so this lesson runs on its own (lesson 6 shows the proper way):
df["region"] = df["region"].str.title()   # a few rows use lower case
df = df.dropna()                            # drop the few rows with missing values
print(df.shape)

In [ ]:
df.head()

## Features and target

The **target** (often called `y`) is the column you want to predict. The **features**
(`X`) are the columns the model uses to predict it. Here we predict `next_year_spend`
from a few numeric columns.

Never use the target, or anything calculated from it, as a feature: the model would be
"cheating" and would fail on real data.

In [ ]:
features = ["orders_last_year", "avg_order_value", "spend_last_year",
            "years_as_customer", "on_contract", "support_tickets"]
X = df[features]
y = df["next_year_spend"]
print(X.shape, y.shape)

## Train and test sets

A model is only useful if it works on data it has **not** seen. So we hold back part of
the data as a **test set**, train on the rest, and measure on the test set.
`random_state` fixes the shuffle so results are repeatable.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("train:", X_train.shape, "test:", X_test.shape)

## The scikit-learn pattern

Every scikit-learn model works the same way:

1. **Create** it: `model = SomeModel(settings)`
2. **Fit** it to training data: `model.fit(X_train, y_train)`
3. **Predict**: `model.predict(X_test)`

Start with a **baseline**: a "model" that always predicts the average. Any real model
must do better than this, or it is not learning anything useful.

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train, y_train)
baseline_mae = mean_absolute_error(y_test, baseline.predict(X_test))

model = LinearRegression()
model.fit(X_train, y_train)
model_mae = mean_absolute_error(y_test, model.predict(X_test))

print(f"Baseline error (always the average): {baseline_mae:,.0f} USD")
print(f"Linear regression error:             {model_mae:,.0f} USD")

**Mean absolute error (MAE)** is the average size of a prediction's miss, in the same
units as the target. Lower is better. The linear model's error is far below the
baseline's, so it has learned something real. Lesson 2 looks at it properly.

## Practice

1. How many accounts paid late (`paid_late == 1`)? What share is that?
2. For each of these, say whether it is regression, classification, or clustering:
   (a) predicting an invoice's days to pay, (b) predicting whether a deal closes,
   (c) grouping products by sales pattern.
3. Re-run the split with `test_size=0.3`. How many rows are in each set now?
4. Fit a `DummyRegressor(strategy="median")` and compare its MAE with the mean baseline.

In [ ]:
# Your practice code here

## Summary

- Machine learning learns rules from examples instead of you writing them.
- Regression predicts numbers; classification predicts categories; clustering groups rows.
- Features (`X`) predict the target (`y`); never let the target leak into the features.
- Train on one part of the data and test on data the model has not seen.
- scikit-learn models all follow create, `fit`, `predict`, and must beat a baseline.

## Practice Solutions

In [ ]:
import pandas as pd

df = pd.read_csv("data/customer_accounts.csv")

# Quick tidy-up so this lesson runs on its own (lesson 6 shows the proper way):
df["region"] = df["region"].str.title()   # a few rows use lower case
df = df.dropna()                            # drop the few rows with missing values

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error

# 1.
print(df["paid_late"].sum(), "late,", f"{df['paid_late'].mean():.1%} of accounts")

# 2. (a) regression, (b) classification, (c) clustering

# 3.
features = ["orders_last_year", "avg_order_value", "spend_last_year",
            "years_as_customer", "on_contract", "support_tickets"]
X, y = df[features], df["next_year_spend"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print(len(X_train), "train rows,", len(X_test), "test rows")

# 4.
for strategy in ["mean", "median"]:
    dummy = DummyRegressor(strategy=strategy).fit(X_train, y_train)
    print(strategy, f"{mean_absolute_error(y_test, dummy.predict(X_test)):,.0f}")